# Reproducing a Meuse Example with GR

This notebook reproduces a small empirical example based on the Meuse dataset using the current `grpy` API.

## Note on coordinates

The original Meuse dataset is provided in projected planar coordinates (`x`, `y`) in **RD New / EPSG:28992**.
The current `GimbalRegression` interface expects geographic coordinates (`lat`, `lon`), so this notebook converts the original coordinates to **WGS84 / EPSG:4326** before fitting.

This conversion is suitable for demonstrating package usage. For benchmark comparisons with methods that operate directly on planar coordinates, the projected coordinates should still be handled carefully in the benchmark code.

## Expected data file

Put your data at `../data/meuse.csv` and make sure it contains at least:

- `x`
- `y`
- `cadmium`
- `lead`
- `zinc`
- `dist`


In [16]:
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pyproj import Transformer
from grpy import GimbalRegression

## Data

### Load data

In [ ]:
data_path = "../data/meuse.csv"
df = pd.read_csv(data_path)
df.head()

### Convert projected coordinates to latitude and longitude

The original Meuse coordinates are in RD New (`EPSG:28992`). We transform them to WGS84 (`EPSG:4326`) for use with the current `grpy` interface.

In [ ]:
transformer = Transformer.from_crs("EPSG:28992", "EPSG:4326", always_xy=True)
lon, lat = transformer.transform(df["x"].to_numpy(), df["y"].to_numpy())
df = df.copy()
df["lon"] = lon
df["lat"] = lat

### Draw a reproducible subsample

This matches the style of a reduced empirical run rather than using all 155 observations.

In [ ]:
random.seed(19)
ixs = random.sample(range(df.shape[0]), 120)
Meuse_samples = (
    df.loc[ixs, ["lat", "lon", "cadmium", "lead"]]
    .dropna()
    .reset_index(drop=True)
)
Meuse_samples.head()

## Model Fitting

In [ ]:
model = GimbalRegression(
    K=30,
    h_m=2000.0,
    kappa=2.0,
    gamma=1.0,
    n0=20.0,
    use_ess=True,
    res_weight_mode="distance",
    compute_local_moran=True,
).fit(
    y=Meuse_samples["cadmium"].to_numpy(dtype=np.float64),
    x=Meuse_samples["lead"].to_numpy(dtype=np.float64),
    lat=Meuse_samples["lat"].to_numpy(dtype=np.float64),
    lon=Meuse_samples["lon"].to_numpy(dtype=np.float64),
)


## Inspecting Results

### Full ste of local results

In [ ]:
results = model.results_
results.head()

### Model summary

In [ ]:
model.summary()

### Model diagnostics

In [ ]:
model.diagnostics()

### Model predictions

In [ ]:
y_hat = model.predict()
y_hat

### Distributions of diagnostics

In [ ]:
results["condM_nor"].hist()

## Visualization

### Coefficient maps

In [ ]:
model.draw_map(
    column="B1",
    title="Local coefficient B1 (lead effect)",
    basemap=True,
    show=True,
)

### Diagnostic maps

#### Condition number

In [ ]:
model.draw_map(
    column="condM_nor",
    title="Condition number",
    basemap=True,
    show=True,
)

#### Effective sample size

In [ ]:
model.draw_map(
    column="neff_post",
    title="Post-correction ESS",
    basemap=False,
    show=True,
)

#### Residuals

In [ ]:
model.draw_map(
    column="resid",
    title="Residuals",
    basemap=True,
    show=True,
)